In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# Cell — Rolling-origin backtest, written out explicitly
#
# Random train/test splitting leaks future information into training and makes
# any resulting score meaningless. A forecast must be evaluated the way it is
# used: standing at week t, seeing only weeks 1..t, predicting t+1..t+H.
#
# Each iteration below reproduces one replenishment decision. With 135 weeks,
# a minimum training window of 78 and H = 3, this yields ~55 scored decisions
# per store per model.

import numpy as np
import pandas as pd

H = config.HORIZON            # 3 weeks = review period + lead time
MIN_TRAIN = 78                # 1.5 years, enough for a 52-week seasonal lookback

def seasonal_naive_forecast(history, h, m=52):
    """Same week last year. A deliberately strong baseline for retail."""
    if len(history) < m:
        return np.repeat(history.iloc[-1], h)
    return np.array([history.iloc[-m + (i % m)] for i in range(h)])

results = []

for store, g in weekly.groupby("Store"):
    s = g.sort_values("Week").set_index("Week")["Sales"].astype(float)

    for t in range(MIN_TRAIN, len(s) - H + 1):
        history = s.iloc[:t]          # everything known at the decision point
        actual = s.iloc[t:t + H]      # what actually happened, unseen at time t

        pred = seasonal_naive_forecast(history, H)

        for i in range(H):
            results.append({
                "store": store,
                "origin": s.index[t - 1],
                "target_date": actual.index[i],
                "h": i + 1,
                "y_true": actual.iloc[i],
                "y_pred": pred[i],
            })

bt = pd.DataFrame(results)
bt["error"] = bt["y_pred"] - bt["y_true"]     # positive = over-forecast

print(f"Origins per store: {bt.groupby('store')['origin'].nunique().to_dict()}")
print(f"Total scored predictions: {len(bt):,}\n")

print("Seasonal naive accuracy by store and horizon:")
print(
    bt.groupby(["store", "h"])
      .apply(lambda x: pd.Series({
          "RMSE": np.sqrt((x["error"] ** 2).mean()),
          "MAPE_%": (x["error"].abs() / x["y_true"]).mean() * 100,
          "bias_%": x["error"].mean() / x["y_true"].mean() * 100,
      }), include_groups=False)
      .round(2)
)

KeyError: 'Week'

In [ ]:
print(weekly.dtypes)
print()
print(weekly.head())

Store                  int64
WeekEnding    datetime64[us]
Sales                  int64
PromoShare           float64
OpenDays               int64
dtype: object

   Store WeekEnding  Sales  PromoShare  OpenDays
0    198 2013-01-06   7553        0.00         4
1    198 2013-01-13  25250        0.83         6
2    198 2013-01-20  10544        0.00         6
3    198 2013-01-27  22567        0.83         6
4    198 2013-02-03  11020        0.00         6


In [ ]:
# Cell — Rolling-origin backtest, written out explicitly
#
# Random train/test splitting leaks future information into training and makes
# any resulting score meaningless. A forecast must be evaluated the way it is
# used: standing at week t, seeing only weeks 1..t, predicting t+1..t+H.
#
# Each iteration below reproduces one replenishment decision. The first and last
# weeks are dropped: they are truncated by the dataset boundary (the panel starts
# on a Tuesday and ends on a Thursday), so their totals reflect missing days
# rather than trading behaviour.

import numpy as np
import pandas as pd

H = config.HORIZON
MIN_TRAIN = config.MIN_TRAIN_WEEKS
SEASON = config.SEASON_LENGTH

# Drop boundary-truncated weeks
bounds = weekly.groupby("Store")["WeekEnding"].agg(["min", "max"])
mask = weekly.apply(
    lambda r: r["WeekEnding"] not in (bounds.loc[r["Store"], "min"],
                                      bounds.loc[r["Store"], "max"]),
    axis=1,
)
panel = weekly[mask].copy()
print(f"Weeks retained: {len(panel)} of {len(weekly)}")


def seasonal_naive_forecast(history, h, m=SEASON):
    """Same week last year. A deliberately strong baseline for retail."""
    if len(history) < m:
        return np.repeat(history.iloc[-1], h)
    return np.array([history.iloc[-m + (i % m)] for i in range(h)])


results = []

for store, g in panel.groupby("Store"):
    s = g.sort_values("WeekEnding").set_index("WeekEnding")["Sales"].astype(float)

    for t in range(MIN_TRAIN, len(s) - H + 1):
        history = s.iloc[:t]          # everything known at the decision point
        actual = s.iloc[t:t + H]      # what actually happened, unseen at time t

        pred = seasonal_naive_forecast(history, H)

        for i in range(H):
            results.append({
                "store": store,
                "origin": s.index[t - 1],
                "target_date": actual.index[i],
                "h": i + 1,
                "y_true": actual.iloc[i],
                "y_pred": pred[i],
            })

bt = pd.DataFrame(results)
bt["error"] = bt["y_pred"] - bt["y_true"]     # positive = over-forecast

print(f"Origins per store: {bt.groupby('store')['origin'].nunique().to_dict()}")
print(f"Total scored predictions: {len(bt):,}\n")

print("Seasonal naive accuracy by store and horizon:")
print(
    bt.groupby(["store", "h"])
      .apply(lambda x: pd.Series({
          "RMSE": np.sqrt((x["error"] ** 2).mean()),
          "MAPE_%": (x["error"].abs() / x["y_true"]).mean() * 100,
          "bias_%": x["error"].mean() / x["y_true"].mean() * 100,
      }), include_groups=False)
      .round(2)
)

Weeks retained: 266 of 270
Origins per store: {198: 53, 733: 53}
Total scored predictions: 318

Seasonal naive accuracy by store and horizon:
            RMSE  MAPE_%  bias_%
store h                         
198   1 5,703.76   23.91   -0.44
      2 5,707.35   23.94   -0.34
      3 5,698.87   23.90   -0.75
733   1 9,463.91    7.14   -3.29
      2 9,476.04    7.16   -3.10
      3 9,472.31    7.15   -3.01


In [ ]:
# Cell — Cumulative forecast error over the protection period
#
# Safety stock does not absorb weekly error; it absorbs the error of *cumulative*
# demand across the protection period P = R + L. The two aggregate differently:
# independent noise grows with sqrt(P), while bias grows linearly with P. A model
# that is noisy but unbiased can therefore need less safety stock than one that
# looks more accurate week by week but drifts in one direction.

cum = (
    bt.groupby(["store", "origin"])
      .agg(forecast_P=("y_pred", "sum"), actual_P=("y_true", "sum"))
      .reset_index()
)
cum["error_P"] = cum["forecast_P"] - cum["actual_P"]

summary = cum.groupby("store").agg(
    mean_demand_P=("actual_P", "mean"),
    bias_P=("error_P", "mean"),
    noise_std_P=("error_P", "std"),
)
summary["bias_%"] = summary["bias_P"] / summary["mean_demand_P"] * 100
summary["noise_%"] = summary["noise_std_P"] / summary["mean_demand_P"] * 100

# If weekly errors were independent, cumulative std would be weekly_std * sqrt(P).
weekly_std = bt.groupby("store")["error"].std()
summary["expected_if_independent"] = weekly_std * np.sqrt(H)
summary["autocorrelation_factor"] = (
    summary["noise_std_P"] / summary["expected_if_independent"]
)

print("Cumulative error over the 3-week protection period:")
print(summary.round(2))

Cumulative error over the 3-week protection period:
       mean_demand_P     bias_P  noise_std_P  bias_%  noise_%  \
store                                                           
198        50,685.42    -258.53     5,624.61   -0.51    11.10   
733       322,616.09 -10,109.98    17,374.41   -3.13     5.39   

       expected_if_independent  autocorrelation_factor  
store                                                   
198                   9,908.53                    0.57  
733                  15,378.64                    1.13  


In [ ]:
# Cell — Price the gap between the textbook formula and reality
#
# The standard safety stock formula SS = z * sigma_weekly * sqrt(L) assumes
# weekly forecast errors are independent. The autocorrelation factors computed
# above show they are not: store 198's errors cancel across the protection
# period (0.57) while store 733's persist (1.13). Using the formula therefore
# overstocks one store and understocks the other. Bias is treated separately:
# a persistent one-directional error is not uncertainty and should be removed
# by correction, not absorbed by inventory.

from statistics import NormalDist

z = NormalDist().inv_cdf(config.SERVICE_LEVEL)
rate = config.CARRYING_RATE_ANNUAL

out = summary.copy()
out["ss_textbook"] = z * out["expected_if_independent"]
out["ss_empirical"] = z * out["noise_std_P"]
out["ss_error_EUR"] = out["ss_textbook"] - out["ss_empirical"]
out["annual_cost_of_error"] = out["ss_error_EUR"].abs() * rate

# Uncorrected bias must be carried as additional stock on top of safety stock.
out["bias_stock_EUR"] = out["bias_P"].clip(upper=0).abs()
out["annual_cost_of_bias"] = out["bias_stock_EUR"] * rate

print(f"Service level {config.SERVICE_LEVEL:.0%} (z = {z:.3f}), "
      f"carrying rate {rate:.0%}/year\n")
print(out[[
    "ss_textbook", "ss_empirical", "ss_error_EUR", "annual_cost_of_error",
    "bias_stock_EUR", "annual_cost_of_bias",
]].round(0).to_string())

print("\nSafety stock as a share of protection-period demand:")
print((out["ss_empirical"] / out["mean_demand_P"] * 100).round(1).to_string())

Service level 95% (z = 1.645), carrying rate 20%/year

       ss_textbook  ss_empirical  ss_error_EUR  annual_cost_of_error  bias_stock_EUR  annual_cost_of_bias
store                                                                                                    
198      16,298.00      9,252.00      7,046.00              1,409.00          259.00                52.00
733      25,296.00     28,578.00     -3,283.00                657.00       10,110.00             2,022.00

Safety stock as a share of protection-period demand:
store
198   18.30
733    8.90
